# Generative AI — Simple Practical Notes
## Notebook 2: Stable Diffusion, Audio and Multimodal AI

This notebook keeps each workflow straightforward and visible. It uses pretrained models rather than building large generative architectures from scratch.

**Main sections**
1. Generate images with Stable Diffusion.
2. Generate speech from text.
3. Transcribe speech (optional).
4. Ask questions about an image.

**Framework note:** The custom VAE and GAN implementations are in the TensorFlow/Keras notebook. Hugging Face Diffusers' Stable Diffusion pipeline is most mature in PyTorch, so the image-generation section uses PyTorch. The model pipeline is used directly rather than attempting to force an unsupported framework combination.

## 1. Install and import libraries

Use a Google Colab GPU runtime for Stable Diffusion. Run the installation commands only if the packages are not already installed.

In [ ]:
# Uncomment if required in a fresh Colab runtime:
# !pip install -q diffusers transformers accelerate safetensors
# !pip install -q torch torchvision torchaudio soundfile datasets pillow matplotlib

import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

device = "cuda" if torch.cuda.is_available() else "cpu"

print("PyTorch:", torch.__version__)
print("Device:", device)

# Part A — Stable Diffusion

## 2. Load a pretrained Stable Diffusion model

Stable Diffusion uses several components:
- A text encoder to understand the prompt.
- A denoising network to predict and remove noise.
- A scheduler to manage the denoising steps.
- A VAE decoder to convert the final latent representation into an image.

Some model repositories require a Hugging Face account, license acceptance, or authentication. Review the model card and access requirements before loading.

In [ ]:
from diffusers import StableDiffusionPipeline

model_id = "stable-diffusion-v1-5/stable-diffusion-v1-5"

# Use half precision on GPU to reduce memory usage.
if device == "cuda":
    data_type = torch.float16
else:
    data_type = torch.float32

pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=data_type
)

pipe = pipe.to(device)

print("Stable Diffusion model loaded.")

## 3. Generate an image

A prompt is the text instruction describing the image to generate.

The number of inference steps controls how many denoising steps are used. The guidance scale controls how strongly the output follows the prompt.

In [ ]:
prompt = "A red bicycle parked beside a tree, realistic photography, morning sunlight"

image_result = pipe(
    prompt,
    num_inference_steps=25,
    guidance_scale=7.0
)

generated_image = image_result.images[0]

plt.imshow(generated_image)
plt.axis("off")
plt.show()

## 4. Experiment with prompts

Try changing the prompt while keeping the generation settings the same.

For example:
- A watercolor painting of a mountain village.
- A realistic photograph of a mountain village.
- A 3D cartoon illustration of a mountain village.

Observe how the wording changes the generated image.

In [ ]:
prompt = "A watercolor painting of a small mountain village at sunrise"

image_result = pipe(
    prompt,
    num_inference_steps=25,
    guidance_scale=7.0
)

plt.imshow(image_result.images[0])
plt.axis("off")
plt.show()

## 5. Understand negative prompts and settings

A negative prompt describes things we want the model to avoid. It can be useful for discouraging unwanted qualities, but it does not guarantee their absence.

In [ ]:
prompt = "A clean product photograph of a ceramic coffee mug on a wooden table"
negative_prompt = "blurry, low quality, distorted, watermark, extra objects"

image_result = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=30,
    guidance_scale=7.5
)

plt.imshow(image_result.images[0])
plt.axis("off")
plt.show()

# Part B — Text-to-Speech

## 6. Load a pretrained text-to-speech model

The example uses SpeechT5 through Hugging Face Transformers. SpeechT5 requires a speaker embedding for speech generation, so we load one from the accompanying speaker-embedding dataset.

The first run downloads the model and dataset.

In [ ]:
from transformers import pipeline
from datasets import load_dataset

tts = pipeline(
    "text-to-speech",
    model="microsoft/speecht5_tts",
    device=0 if device == "cuda" else -1
)

speaker_data = load_dataset(
    "Matthijs/cmu-arctic-xvectors",
    split="validation"
)

speaker_embedding = torch.tensor(
    speaker_data[7306]["xvector"]
).unsqueeze(0)

print("Text-to-speech model loaded.")

## 7. Convert text into speech

Pass text and the speaker embedding to the model. The result contains an audio array and its sampling rate.

In [ ]:
text = "Generative AI models can create text, images, audio and other forms of content."

speech_result = tts(
    text,
    forward_params={"speaker_embeddings": speaker_embedding}
)

audio = speech_result["audio"]
sampling_rate = speech_result["sampling_rate"]

print("Sampling rate:", sampling_rate)
print("Audio shape:", np.asarray(audio).shape)

## 8. Play and save the generated audio

In [ ]:
from IPython.display import Audio, display

display(Audio(audio, rate=sampling_rate))

In [ ]:
import soundfile as sf

sf.write(
    "generated_speech.wav",
    np.asarray(audio).squeeze(),
    sampling_rate
)

print("Audio saved as generated_speech.wav")

# Part C — Speech-to-Text

## 9. Transcribe an audio file (optional)

Speech-to-text converts spoken audio into written text. This example uses the Whisper model.

The cell below loads the model. Set the audio file path to a WAV or other supported audio file before running the transcription lines.

In [ ]:
asr = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-tiny",
    device=0 if device == "cuda" else -1
)

# Use an audio file available in the notebook environment.
# audio_file = "generated_speech.wav"
# transcription = asr(audio_file)
# print(transcription["text"])

# Part D — Multimodal AI

## 10. Load an image-question-answering model

A vision-language model accepts an image and a text question. It uses visual information to produce an answer.

The example uses BLIP for visual question answering.

In [ ]:
image_qa = pipeline(
    "visual-question-answering",
    model="Salesforce/blip-vqa-base",
    device=0 if device == "cuda" else -1
)

print("Image question-answering model loaded.")

## 11. Create a sample image

We create a simple image locally so the example does not depend on an external image URL. You can replace this with an image you have permission to use.

In [ ]:
sample_image = Image.new("RGB", (500, 320), "white")
draw = ImageDraw.Draw(sample_image)

draw.rectangle((0, 220, 500, 320), fill=(190, 220, 180))
draw.ellipse((80, 70, 220, 210), fill=(240, 190, 70), outline="black", width=3)
draw.rectangle((280, 130, 390, 230), fill=(100, 160, 220), outline="black", width=3)
draw.line((280, 130, 335, 80), fill="black", width=5)
draw.line((390, 130, 335, 80), fill="black", width=5)

display(sample_image)

## 12. Ask a question about the image

Pass the image and a question to the model. Try different questions and compare the answers.

In [ ]:
question = "What objects are visible in this image?"

answer = image_qa(
    image=sample_image,
    question=question
)

print(answer)

## 13. Try another image

Replace the sample image with a local image. The model can be asked to identify objects, describe a scene, or answer questions about visible content.

Important information should always be checked against the original image because vision-language models can produce incorrect answers.

In [ ]:
# Example:
# my_image = Image.open("/content/my_image.jpg").convert("RGB")
# display(my_image)
# print(image_qa(image=my_image, question="Describe the scene."))

# 14. Responsible use

Before using these models in a real application:

- Check model licenses and usage restrictions.
- Review generated images for unsafe content, bias, and visual errors.
- Obtain permission before imitating or cloning someone's voice.
- Avoid deceptive impersonation or misleading synthetic media.
- Verify important information extracted from images.
- Avoid sending sensitive content to services without authorization.
- Keep appropriate safety checks, authentication, rate limits, and logging in deployed applications.

## Quick review
1. What does the text encoder do in Stable Diffusion?
2. What is the role of the denoising network?
3. What does the scheduler control?
4. What is the difference between text-to-speech and speech-to-text?
5. How does a vision-language model use both an image and a question?
6. Why should model-generated answers be verified?